In [1]:
from qiskit import QuantumCircuit
path = r"C:\Users\joven\Documents\MIT Iqhack 26\Files\P5_soft_rise.qasm"
qc = QuantumCircuit.from_qasm_file(path)

print("qubits:", qc.num_qubits, "depth:", qc.depth(), "ops:", qc.size())
print(qc.count_ops())

import bluequbit
bq = bluequbit.init("luES7Zu4Z7qgi5qkP7Qp29zZYYgvYjbA")


[BQ-PYTHON-SDK][WARNING] - Beta version 0.18.1b1 of BlueQubit Python SDK is being used.


qubits: 50 depth: 29 ops: 1030
OrderedDict([('u3', 703), ('cz', 327)])


In [ ]:

import numpy as np
from collections import Counter
from qiskit_aer import AerSimulator


# 0) Simulator

def make_mps_simulator(bond=64, trunc=5e-3):
    sim = AerSimulator(method="matrix_product_state")
    sim.set_options(
        matrix_product_state_max_bond_dimension=int(bond),
        matrix_product_state_truncation_threshold=float(trunc),
    )
    return sim

def ensure_measured(qc):
    tqc = qc.copy()
    if tqc.num_clbits == 0:
        tqc.measure_all()
    return tqc

def sample_counts(sim, qc, shots=20000, seed=0):
    tqc = ensure_measured(qc)
    job = sim.run(tqc, shots=int(shots), seed_simulator=int(seed))
    return job.result().get_counts()


# 1) m_i and C_ij from 

def stats_from_counts(counts, n):
    total = sum(counts.values())
    m = np.zeros(n, dtype=np.float64)
    C = np.zeros((n, n), dtype=np.float64)

    for bstr, c in counts.items():
        bstr = bstr.zfill(n)  # q[n-1]..q[0]
        s = np.fromiter((1.0 if ch == "0" else -1.0 for ch in bstr),
                        count=n, dtype=np.float64)
        m += c * s
        C += c * np.outer(s, s)

    m /= total
    C /= total
    return m, C

def pick_top_pairs(C, topk=1200):
    n = C.shape[0]
    pairs = []
    for i in range(n):
        for j in range(i+1, n):
            pairs.append((abs(C[i, j]), i, j, C[i, j]))
    pairs.sort(reverse=True, key=lambda t: t[0])
    return pairs[:topk]


# 2) Ising MAP solve (ICM)

def solve_icm(h, edges, restarts=160, sweeps=60, rng_seed=0):
    n = len(h)
    rng = np.random.default_rng(rng_seed)
    adj = [[] for _ in range(n)]
    for i, j, J in edges:
        adj[i].append((j, J))
        adj[j].append((i, J))

    def score(s):
        val = float((h * s).sum())
        for i, j, J in edges:
            val += float(J * s[i] * s[j])
        return val

    best_s, best_val = None, -1e100
    for _ in range(restarts):
        s = np.sign(h + 0.12 * rng.standard_normal(n))
        s[s == 0] = 1
        for _ in range(sweeps):
            changed = False
            for i in range(n):
                field = h[i] + sum(J * s[j] for j, J in adj[i])
                new_si = 1 if field >= 0 else -1
                if new_si != s[i]:
                    s[i] = new_si
                    changed = True
            if not changed:
                break
        val = score(s)
        if val > best_val:
            best_val, best_s = val, s.copy()
    return best_s, best_val

def s_to_bitstring(s):
    bits = (s < 0).astype(np.uint8)  # -1 -> bit 1
    return ''.join('1' if b else '0' for b in bits.tolist())

def make_candidates(best_s, h, k_uncertain=16, cap=4096):
    uncertain = np.argsort(np.abs(h))[:k_uncertain]
    cands = set([s_to_bitstring(best_s)])
    m = len(uncertain)
    for mask in range(1, 1 << m):
        if len(cands) >= cap:
            break
        s2 = best_s.copy()
        for t in range(m):
            if (mask >> t) & 1:
                s2[uncertain[t]] *= -1
        cands.add(s_to_bitstring(s2))
    return sorted(cands), uncertain

def bstr_to_words(bstr):
    x = int(bstr, 2)
    nbytes = (len(bstr) + 7) // 8
    b = x.to_bytes(nbytes, byteorder="big", signed=False)
    pad = (-len(b)) % 8
    if pad:
        b = (b"\x00" * pad) + b
    return np.frombuffer(b, dtype=np.uint64)

def popcount_words_matrix(x):
    # x: (U, L) uint64
    y = x.copy()
    y = y - ((y >> 1) & np.uint64(0x5555555555555555))
    y = (y & np.uint64(0x3333333333333333)) + ((y >> 2) & np.uint64(0x3333333333333333))
    y = (y + (y >> 4)) & np.uint64(0x0F0F0F0F0F0F0F0F)
    return ((y * np.uint64(0x0101010101010101)) >> 56).sum(axis=1).astype(np.int64)

def prepare_packed_counts(counts, n):
    # Pre-pack observed bitstrings once
    keys = list(counts.keys())
    vals = np.array([counts[k] for k in keys], dtype=np.int64)
    packed = [bstr_to_words(k.zfill(n)) for k in keys]
    L = max(w.size for w in packed)
    packed = [np.pad(w, (L - w.size, 0)) for w in packed]
    packed = np.stack(packed, axis=0)  # (U, L)
    return keys, vals, packed, L

def rank_by_hamming_ball_from_counts(prep, candidates, r=4, topk_print=10):
    keys, vals, packed, L = prep
    ranked = []
    for c in candidates:
        wc = bstr_to_words(c)
        wc = np.pad(wc, (L - wc.size, 0))
        d = popcount_words_matrix(np.bitwise_xor(packed, wc))
        score = int(vals[d <= r].sum())
        ranked.append((c, score))
    ranked.sort(key=lambda t: t[1], reverse=True)
    print(f"\nTop-{topk_print} by Hamming-ball mass (r={r}):")
    for c, sc in ranked[:topk_print]:
        print(sc, c)
    return ranked

BOND = 64
TRUNC = 5e-3

# model shots
RUNS = 10
SHOTS_PER_RUN = 1000000     # total model shots = 2,000,000 (already plenty)

# Ising + candidate search
TOP_PAIRS = 1200
RESTARTS = 200
SWEEPS = 70
K_UNCERTAIN = 16
CAND_CAP = 4096

# verification (counts-based, fast)
VERIFY_SHOTS = 10000000      # raise to 1_000_000 if the top scores tie
RADIUS_LIST = [2, 3, 4, 5]

n = qc.num_qubits
print("qubits=", n, "depth=", qc.depth(), "ops=", qc.count_ops())

sim = make_mps_simulator(bond=BOND, trunc=TRUNC)

# (A) Build low-order stats from multiple runs (counts only)
agg = Counter()
for r in range(RUNS):
    c = sample_counts(sim, qc, shots=SHOTS_PER_RUN, seed=r)
    agg.update(c)
    print(f"  run {r+1}/{RUNS}: unique={len(c)}")
agg = dict(agg)

h, C = stats_from_counts(agg, n)
print("sanity max|m|:", float(np.max(np.abs(h))), " (<=1)")
print("median|m|:", float(np.median(np.abs(h))), "min|m|:", float(np.min(np.abs(h))))

# (B) Ising edges
pairs = pick_top_pairs(C, topk=TOP_PAIRS)
edges = []
for _, i, j, Cij in pairs:
    Cij = float(np.clip(Cij, -0.95, 0.95))
    J = 0.5 * np.log((1 + Cij) / (1 - Cij))  # atanh
    edges.append((i, j, J))

# (C) MAP solve
best_s, best_val = solve_icm(h, edges, restarts=RESTARTS, sweeps=SWEEPS)
base = s_to_bitstring(best_s)
print("\nMAP base:", base)
print("MAP score:", float(best_val))




qubits= 50 depth= 29 ops= OrderedDict([('u3', 703), ('cz', 327)])
  run 1/10: unique=998093
  run 2/10: unique=998150
  run 3/10: unique=998158
  run 4/10: unique=998105
  run 5/10: unique=998093
  run 6/10: unique=998124
  run 7/10: unique=998094
  run 8/10: unique=998108
  run 9/10: unique=998150
  run 10/10: unique=998045
sanity max|m|: 0.9147074  (<=1)
median|m|: 0.5100372 min|m|: 0.0212836

MAP base: 01101000100100001010101011100010010111100011111110
MAP score: 376.268590555601
candidates: 4096 uncertain idx: [47, 30, 44, 31, 15, 7, 36, 1, 14, 24, 41, 13, 37, 16, 19, 26]

Sampling 10000000 verify shots (counts only)...


KeyboardInterrupt: 

In [ ]:


import numpy as np
from collections import Counter

def bstr_to_words(bstr):
    x = int(bstr, 2)
    nbytes = (len(bstr) + 7) // 8
    b = x.to_bytes(nbytes, byteorder="big", signed=False)
    pad = (-len(b)) % 8
    if pad:
        b = (b"\x00" * pad) + b
    return np.frombuffer(b, dtype=np.uint64)

def popcount_words_matrix(x):
    y = x.copy()
    y = y - ((y >> 1) & np.uint64(0x5555555555555555))
    y = (y & np.uint64(0x3333333333333333)) + ((y >> 2) & np.uint64(0x3333333333333333))
    y = (y + (y >> 4)) & np.uint64(0x0F0F0F0F0F0F0F0F)
    return ((y * np.uint64(0x0101010101010101)) >> 56).sum(axis=1).astype(np.int64)

# 1) Take only the weakest bits
k_uncertain_fast = 10
uncertain = np.argsort(np.abs(h))[:k_uncertain_fast]

# 2) Generate small candidate neighborhood
def generate_candidates(base, uncertain, cap=256):
    cands = []
    s0 = np.array([1 if b=="0" else -1 for b in base])
    for mask in range(1 << len(uncertain)):
        if len(cands) >= cap:
            break
        s = s0.copy()
        for i in range(len(uncertain)):
            if (mask >> i) & 1:
                s[uncertain[i]] *= -1
        b = ''.join('1' if x==-1 else '0' for x in s)
        cands.append(b)
    return cands

candidates_fast = generate_candidates(base, uncertain, cap=256)
print("Fast candidates:", len(candidates_fast))

# 3) Prepare packed counts ONCE
keys = list(agg.keys())
vals = np.array([agg[k] for k in keys], dtype=np.int64)

packed = [bstr_to_words(k) for k in keys]
L = max(w.size for w in packed)
packed = [np.pad(w, (L - w.size, 0)) for w in packed]
packed = np.stack(packed, axis=0)

# 4) Hamming-ball score at r=3 only
def hamming_ball_score(cand, r=3):
    wc = bstr_to_words(cand)
    wc = np.pad(wc, (L - wc.size, 0))
    d = popcount_words_matrix(np.bitwise_xor(packed, wc))
    return int(vals[d <= r].sum())

scores = [(c, hamming_ball_score(c, r=3)) for c in candidates_fast]
scores.sort(key=lambda t: t[1], reverse=True)

print("\nTop-10 candidates (r=3):")
for c, sc in scores[:10]:
    print(sc, c)

print("\nFINAL BEST GUESS:", scores[0][0])


Fast candidates: 256

Top-10 candidates (r=3):
15368 01101000100100001010101011100000010111100011111110
14242 01101000100100001010101011100000010111100011011110
13591 01101000100100001010101011100010010111100011111110
13275 01101000100100011010101011100000010111100011111110
12886 01101000100100001010101011100000010111100011111010
12598 01101000100100001010101011100010010111100011011110
12372 01101000100100001010101011100011010111100011111110
12117 01101000100100011010101011100000010111100011011110
11951 01101000100100001010101011100000010111100011011010
11675 01101000100100011010101011100010010111100011111110

FINAL BEST GUESS: 01101000100100001010101011100000010111100011111110
